In [1]:
import os

from loadData import data_pipe
os.environ['CUDA_VISIBLE_DEVICES'] = ','.join(map(str, [0]))
print('using GPU %s' % ','.join(map(str, [0])))

import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import SequentialLR, StepLR, LambdaLR
from thop import profile, clever_format

import csv
import time
import json
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
# plt.rc('font',family='Times New Roman') 

from option import opt
from loadData import data_pipe
from loadData.split_data import HyperX2, HyperX3
from loadData.dataAugmentation import DataAugmentation

from models import resNet2, modules2, vision_transformer, automaticWeightedLoss, CNNBase
# vision_transformer_dino
from models.MS2CANet import pymodel
# from models.Morphs import morphFormer2
from utils import trainer, tester, infoNCE, tools, focalLoss
from utils import mutual_info, distillation, CELoss

# from utils import orthogonalLoss

using GPU 0


In [2]:
args = opt.get_args()
# args.dataset_name = "IndianPines"
# args.dataset_name = "PaviaU"
args.dataset_name = "Houston_2013"
# args.dataset_name = "Houston_2018"
# args.dataset_name = "Berlin"
args.path_data = "/home/icclab/Documents/lqw/DatasetMMF"

# args.backbone = "MiViT" 
args.backbone = "resNet2"
# args.backbone = "resNet4"
# args.backbone = "morphFormer2"
# args.backbone = "vit_dino_s"
# args.backbone = "vit_dino_b"
# args.local_crops_number = 0

# args.split_type = "disjoint"
args.split_type = "ratio"

print("args.backbone", args.backbone)
# print("args.randomCrop", args.randomCrop)

args.backbone resNet2


In [3]:
args.epochs = 50
args.batch_size = 64
args.patch_size = 13
args.randomCrop = 11

# args.patch_size = 14
# args.randomCrop = 12


# args.patch_size = 6
# args.randomCrop = 4
args.pca = True
args.components = 15
# args.pca = False
# args.components = 0

args.lb_smooth = 0.1
args.learning_rate = 0.001
# args.learning_rate = 0.0005
args.weight_decay = 1e-3
args.lambda_contra = 0.1
args.lambda_orth = 0.1
args.lambda_kl = 0
args.lambda_super = 1
args.schedule = True
args.step_size = 30
args.gamma = 0.7

In [4]:
args.print_data_info = False
args.data_info_start = 1
args.show_gt = False
args.remove_zero_labels = True
args.train_ratio = 1

# create dataloader
img1, img2, train_gt, _, _, data_gt, GT = data_pipe.get_data(args)
transform = DataAugmentation(args)
# transform = None

args.contrastive = True
contrastive_dataset = HyperX2(img1, data2=img2, gt=train_gt, transform=transform, args=args)
contrastive_loader = DataLoader(contrastive_dataset, batch_size=args.batch_size, shuffle=True, drop_last=True)


# data_pipe.set_deterministic(seed = 666)
args.print_data_info = True
args.data_info_start = 1
args.show_gt = False
args.remove_zero_labels = True
args.train_ratio = 0.9

# create dataloader
img1, img2, train_gt, val_gt, test_gt, data_gt, GT = data_pipe.get_data(args)
band1 = img1.shape[2]
band2 = img2.shape[2]
args.min = str(np.min(img1))
args.max = str(np.max(img1))
print("img1", img1.shape, "img2", img2.shape, band1, band2, args.min, args.max)
print("train_gt", train_gt.shape, \
		"test_gt", test_gt.shape, \
		"data_gt", data_gt.shape, \
		"GT", GT.shape)
# transform = None

# 不拆分高光谱图像的图像处理
# args.mix = True
args.contrastive = False
train_dataset = HyperX2(img1, data2=img2, gt=train_gt, transform=None, args=args)
val_dataset = HyperX2(img1, data2=img2, gt=val_gt, transform=None, args=args)
test_dataset = HyperX2(img1, data2=img2, gt=test_gt, transform=None, args=args)

# 用于 focalloss
train_gt_pure = train_gt[train_gt > 0] - 1
test_gt_pure = test_gt[test_gt > 0] - 1
loss_weight = focalLoss.loss_weight_calculation(test_gt_pure)

train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False)

class_num = np.max(train_gt)
print(class_num, train_gt.shape, len(train_loader.dataset))

pca is used
pca is used
print_data_info : ---->
class 1 	 178 	 20 	 1053
class 2 	 171 	 19 	 1064
class 3 	 173 	 19 	 505
class 4 	 169 	 19 	 1056
class 5 	 167 	 19 	 1056
class 6 	 164 	 18 	 143
class 7 	 176 	 20 	 1072
class 8 	 172 	 19 	 1053
class 9 	 174 	 19 	 1059
class 10 	 172 	 19 	 1036
class 11 	 163 	 18 	 1054
class 12 	 173 	 19 	 1041
class 13 	 166 	 18 	 285
class 14 	 163 	 18 	 247
class 15 	 168 	 19 	 473
total     	 2549 	 283 	 12197
img1 (363, 1919, 15) img2 (363, 1919, 1) 15 1 -61.105827 69.57207
train_gt (363, 1919) test_gt (363, 1919) data_gt (363, 1919) GT (349, 1905)
15 (363, 1919) 2549


In [5]:
for x11, x12, x21, x22, y in contrastive_loader:
    print("x11.shape, x12.shape", x11.shape, x12.shape, len(x11))
    print("x21.shape, x22.shape", x21.shape, x22.shape, len(x21))

    # fig, axes = plt.subplots(3, 3, figsize=(9, 9))
    # for i in range(9):
    #     ax = axes[i // 3, i % 3]
    #     img = x11[i].permute(1, 2, 0)[:, :, :3]  # 转为HWC格式，并取前3通道
    #     img = (img - img.min()) / (img.max() - img.min() + 1e-6)  # 归一化以避免显示错误
    #     ax.imshow(img)
    #     ax.axis('off')
        
    # plt.tight_layout(pad=0)  # 去除额外空白边距
    # plt.show()
    break

for x11, x21, y in train_loader:
    print("x11.shape", x11.shape)
    print("x21.shape", x21.shape)
    break

x11.shape, x12.shape torch.Size([64, 15, 11, 11]) torch.Size([64, 1, 11, 11]) 64
x21.shape, x22.shape torch.Size([64, 15, 11, 11]) torch.Size([64, 1, 11, 11]) 64
x11.shape torch.Size([64, 15, 13, 13])
x21.shape torch.Size([64, 1, 13, 13])


# Path

In [6]:
# 加载已有权重路径
# args.result_dir = "/home/leo/Multimodal_Classification/KnowCLPlus/results_D8/04-13-18-04-mine_cnn_D8"

args.result_dir = os.path.join("/home/icclab/Documents/lqw/Multimodal_Classification/MoEIF/result",
                    datetime.now().strftime("%m-%d-%H-%M-" + args.backbone + "_" + args.dataset_name))
print(args.result_dir)

if not os.path.exists(args.result_dir):
    os.mkdir(args.result_dir)
with open(args.result_dir + '/args.json', 'w') as fid:
    json.dump(args.__dict__, fid, indent=2)

/home/icclab/Documents/lqw/Multimodal_Classification/MoEIF/result/08-01-17-12-resNet2_Houston_2013


# model

In [7]:
if args.backbone == "resNet2":
    encoder = CNNBase.Model_base(channell=band1, channel2=band2).to(args.device)
    args.feature_dim = 512
    # args.feature_dim = 2048

# elif args.backbone == "vit_dino":
#     # encoder = vision_transformer_dino.vit_hsi(band1, args.randomCrop, args.vit_patch).cuda()
#     # vision_transformer_dino.load_transformer(encoder, "/home/icclab/Documents/lqw/Multimodal_Classification/KnowCL_competitive/weights/dinov2_vits14_pretrain.pth")
#     # args.feature_dim = 384

#     selected_layers = [0, 2, 5, 7]  # 从 dinov2 中选择的层
#     encoder = vision_transformer_dino.vit_selected_hsi(band1, args.randomCrop, args.vit_patch, selected_layers=selected_layers).to(args.device)
#     vision_transformer_dino.load_selected_blocks(encoder, \
#                     "/home/icclab/Documents/lqw/Multimodal_Classification/KnowCLPlus/weights/dinov2_vits14_pretrain.pth",
#                     selected_layers=selected_layers)
#     args.feature_dim = 384

#     # selected_layers = [0, 2, 5, 7]  # 从 dinov2 中选择的层
#     # encoder = vision_transformer_dino.vit_selected_base(band1, args.randomCrop, args.vit_patch, selected_layers=selected_layers).to(args.device)
#     # vision_transformer_dino.load_selected_blocks(encoder, \
#     #                 "/home/icclab/Documents/lqw/Multimodal_Classification/KnowCLPlus/weights/dinov2_vitb14_pretrain.pth", selected_layers)
#     # args.feature_dim = 768

# elif args.backbone == "morphFormer2":
#     args.FM = 16
#     encoder = morphFormer2.MorphFormer(args.FM, band1, band2, class_num, args.randomCrop).to(args.device)
#     args.feature_dim = 64

super_head = modules2.FDGCHead(args.feature_dim, class_num=class_num).to(args.device)
contra_head = modules2.DINOHead(args.feature_dim).to(args.device)
awl = automaticWeightedLoss.AutomaticWeightedLoss(3).to(args.device)
params = list(super_head.parameters()) + list(encoder.parameters()) + list(contra_head.parameters())

optimizer = torch.optim.AdamW(params, lr=args.learning_rate, weight_decay=args.weight_decay)
warmup = LambdaLR(optimizer, lr_lambda=lambda e: min(1.0, e / 5.0))
decay = StepLR(optimizer, step_size=args.step_size, gamma=args.gamma)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=args.step_size, gamma=args.gamma)  # 学习太快
scheduler = SequentialLR(optimizer, schedulers=[warmup, decay], milestones=[5])

criterion1 = infoNCE.InfoNCE().to(args.device)
criterion2 = CELoss.LabelSmoothSoftmaxCEV1(lb_smooth=args.lb_smooth).to(args.device)
# criterion2 = torch.nn.CrossEntropyLoss().to(args.device)
# criterion3 = distillation.KL_loss().to(args.device)
# criterion4 = mutual_info.Mutual_info_cnn(args.feature_dim, args.feature_dim).to(args.device)
# criterion4 = orthogonalLoss.OrthogonalLoss(version='dot', sample_size=20).to(args.device)
loss_weight = loss_weight.to(args.device)
criterion5 = focalLoss.FocalLoss(loss_weight, gamma=2, alpha=None).to(args.device)

In [8]:
# flops, params = profile(net, 
#                         inputs=(torch.randn(2, 
#                                         args.components, 
#                                         args.patch_size, 
#                                         args.patch_size).cuda(),))
# flops, params = clever_format([flops, params])
# print('# Model Params: {} FLOPs: {}'.format(params, flops))

# # contra_head
# flops, params = profile(contra_head, inputs=(torch.randn(1, args.feature_dim).cuda(),))
# flops, params = clever_format([flops, params])
# print('# Model Params: {} FLOPs: {}'.format(params, flops))

# # super_head
# flops, params = profile(super_head, inputs=(torch.randn(1, args.feature_dim).cuda(),))
# flops, params = clever_format([flops, params])
# print('# Model Params: {} FLOPs: {}'.format(params, flops))

# 训练

In [9]:
def run_iteration(args):
    best_loss = 999
    best_acc = 0
    train_losses = []
    test_losses = []
    loss_contras = []
    loss_orths = []
    loss_supers = []
    train_accuracies = []
    test_accuracies = []

    start_time = time.time()
    for epoch in range(0, args.epochs):
    ###################################################################->>>>>>>>>

        if args.backbone == "resNet2" or args.backbone == "resNet4" or \
            args.backbone == "vit_dino_s" or args.backbone == "vit_dino_b":
            train_loss, loss_contra, loss_orth, loss_super, loss_dist, train_accuracy, train_time = \
                        trainer.train(encoder, contra_head, super_head, awl, criterion1, \
                                    criterion2, contrastive_loader, train_loader, optimizer, args)
            if epoch % args.log_interval == 0:
                test_loss, test_preds, targets, test_accuracy, test_time = \
                        tester.test_resNet2(encoder, super_head, criterion2, val_loader, args) 

        # elif args.backbone == "morphFormer2":
        #     train_loss, loss_contra, loss_orth, loss_super, loss_dist, train_accuracy, train_time = \
        #                 trainer.train_morphFormer2(encoder, contra_head, super_head, awl, criterion1, \
        #                             criterion2, criterion3, criterion4, contrastive_loader, \
        #                             train_loader, optimizer, args)
        #     if epoch % args.log_interval == 0:
        #         test_loss, test_preds, targets, test_accuracy, test_time = \
        #                 tester.test_morphFormer2(encoder, super_head, criterion2, val_loader, args)
        else:
            raise NotImplementedError("No models")         
            
        print('Train Epoch: [{}/{}] Loss: {:.4f} lco: {:.4f} lor: {:.4f} lsu: {:.4f} ldt: {:.4f} TrainAcc: {:.2f} TestAcc: {:.2f} TIME: {:.4f}'.format(\
                                epoch, args.epochs, 
                                train_loss, 
                                loss_contra, 
                                loss_orth, 
                                loss_super, 
                                loss_dist,
                                train_accuracy, 
                                test_accuracy, 
                                train_time
                                ))

        with open(os.path.join(args.result_dir, "log.csv"), 'a+', encoding='gbk') as f:
            row=[["epoch", epoch,
                "loss", train_loss,
                "test_loss", test_loss,
                "loss_contra", loss_contra,
                "loss_super", loss_super,
                "loss_dist", loss_dist,
                "train_accuracy", train_accuracy,
                "test_accuracy", test_accuracy,
                "train_time", train_time,
                "test_time", test_time,
                '\n']]
            write=csv.writer(f)
            for i in range(len(row)):
                write.writerow(row[i])

        train_losses.append(train_loss)
        test_losses.append(test_loss)
        loss_contras.append(loss_contra)
        loss_orths.append(loss_orth)
        loss_supers.append(loss_super)
        train_accuracies.append(train_accuracy)
        test_accuracies.append(test_accuracy)

        scheduler.step()


        best_loss, best_acc = tools.save_weights(train_loss, test_loss, best_loss, best_acc, \
                                        test_accuracy, epoch, encoder, super_head, contra_head,  optimizer, args)
    
    # torch.cuda.empty_cache()
    Total_train_time = time.time() - start_time
    torch.save({
            "epoch": epoch,
            "base": encoder.state_dict(),
            "contra_head": contra_head.state_dict(),
            "super_head": super_head.state_dict(),
            "optimizer": optimizer.state_dict()}, 
    os.path.join(args.result_dir, "model_last.pth"))


    # --------------------------- 临时的测试-----------------------
    args.resume = os.path.join(args.result_dir, "test_loss.pth")
    if args.resume != '':
        checkpoint = torch.load(args.resume)
        encoder.load_state_dict(checkpoint['model'], strict=False)
        super_head.load_state_dict(checkpoint['super_head'], strict=False)
        contra_head.load_state_dict(checkpoint['contra_head'], strict=False)
        epoch = checkpoint['epoch'] + 1
        print('Loaded from: {} epoch {}'.format(args.resume, epoch))
    else:
        raise ValueError("No resume file found")

    # linear 精度
    if args.backbone == "resNet2" or args.backbone == "resNet4" or \
            args.backbone == "vit_dino_s" or args.backbone == "vit_dino_b":
        test_loss, test_preds, targets, test_acc, test_time = \
            tester.test_resNet2(encoder, super_head, criterion2, test_loader, args) 
    else:
        raise NotImplementedError("No models")
    classification, kappa = tester.get_results(test_preds, targets)
    print(classification)

    # print(KNNOA)
    with open(os.path.join(args.result_dir, "log_final.csv"), 'a+', encoding='gbk') as f:
        row=[["\nLinear Train",
            "\nepoch", epoch, 
            "\nclassification\n", classification,
            "\nkappa", kappa,
            "\nTest_time", round(test_time, 2),
            "\nTrian_time", round(Total_train_time, 2),
            "\n"
            ]]
        write=csv.writer(f)
        for i in range(len(row)):
            write.writerow(row[i])
    # --------------------------- 临时的测试-----------------------

args.number_iterations = 3
for i in range(args.number_iterations):

    if args.backbone == "resNet2":
        encoder = CNNBase.Model_base(channell=band1, channel2=band2).to(args.device)
        args.feature_dim = 512
        # args.feature_dim = 2048

    # elif args.backbone == "resNet4":
    #     encoder = resNet4.Model_base(channell=band1, channel2=band2).to(args.device)
    #     args.feature_dim = 512
    #     # args.feature_dim = 2048

    # elif args.backbone == "vit_dino_s":
    #     selected_layers = [0, 2, 5, 7]  # 从 dinov2 中选择的层
    #     encoder = vision_transformer_dino.Vit_base(band1, band2, args.randomCrop, selected_layers=selected_layers).to(args.device)
    #     args.feature_dim = 384

    # elif args.backbone == "vit_dino_b":
    #     selected_layers = [0, 2, 5, 7]  # 从 dinov2 中选择的层
    #     encoder = vision_transformer_dino.Vit_base(band1, band2, args.randomCrop, selected_layers=selected_layers).to(args.device)
    #     args.feature_dim = 768 

        
    super_head = modules2.FDGCHead(args.feature_dim, class_num=class_num).to(args.device)
    contra_head = modules2.DINOHead(args.feature_dim).to(args.device)
    awl = automaticWeightedLoss.AutomaticWeightedLoss(3).to(args.device)
    params = list(super_head.parameters()) + list(encoder.parameters()) + list(contra_head.parameters())

    optimizer = torch.optim.AdamW(params, lr=args.learning_rate, weight_decay=args.weight_decay)
    warmup = LambdaLR(optimizer, lr_lambda=lambda e: min(1.0, e / 5.0))
    decay = StepLR(optimizer, step_size=args.step_size, gamma=args.gamma)
    # scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=args.step_size, gamma=args.gamma)  # 学习太快
    scheduler = SequentialLR(optimizer, schedulers=[warmup, decay], milestones=[5])

    criterion1 = infoNCE.InfoNCE().to(args.device)
    criterion2 = CELoss.LabelSmoothSoftmaxCEV1(lb_smooth=args.lb_smooth).to(args.device)
    # criterion2 = torch.nn.CrossEntropyLoss().to(args.device)
    # criterion3 = distillation.KL_loss().to(args.device)
    # criterion4 = mutual_info.Mutual_info_cnn(args.feature_dim, args.feature_dim).to(args.device)
    # criterion4 = orthogonalLoss.OrthogonalLoss(version='dot', sample_size=20).to(args.device)
    loss_weight = loss_weight.to(args.device)
    criterion5 = focalLoss.FocalLoss(loss_weight, gamma=2, alpha=None).to(args.device)
    criterion6 = infoNCE.NT_xent_loss_W_EN().to(args.device)

    run_iteration(args)
# Total_train_time = run_iteration(args)

Train Epoch: [0/50] Loss: 4.3139 lco: 1.5440 lor: 0.0000 lsu: 2.7699 ldt: 0.0000 TrainAcc: 6.71 TestAcc: 15.55 TIME: 1.9200
Save model with test_loss: 2.6904, epoch: 0
Train Epoch: [1/50] Loss: 2.3027 lco: 1.1182 lor: 0.0000 lsu: 1.1845 ldt: 0.0000 TrainAcc: 66.42 TestAcc: 84.45 TIME: 1.5600
Save model with test_loss: 1.0533, epoch: 1
Train Epoch: [2/50] Loss: 1.6296 lco: 0.9229 lor: 0.0000 lsu: 0.7067 ldt: 0.0000 TrainAcc: 96.94 TestAcc: 97.53 TIME: 1.5200
Save model with test_loss: 0.7304, epoch: 2
Train Epoch: [3/50] Loss: 1.4662 lco: 0.8634 lor: 0.0000 lsu: 0.6029 ldt: 0.0000 TrainAcc: 99.22 TestAcc: 93.99 TIME: 1.4800
Save model with test_loss: 0.6907, epoch: 3
Train Epoch: [4/50] Loss: 1.4373 lco: 0.8326 lor: 0.0000 lsu: 0.6047 ldt: 0.0000 TrainAcc: 99.73 TestAcc: 93.29 TIME: 1.4500


/home/icclab/miniconda3/envs/leo/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:198: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Train Epoch: [5/50] Loss: 1.3908 lco: 0.7929 lor: 0.0000 lsu: 0.5979 ldt: 0.0000 TrainAcc: 99.69 TestAcc: 89.40 TIME: 1.4700
Save model with test_loss: 0.6350, epoch: 5
Train Epoch: [6/50] Loss: 1.3789 lco: 0.7582 lor: 0.0000 lsu: 0.6207 ldt: 0.0000 TrainAcc: 99.37 TestAcc: 100.00 TIME: 1.5200
Save model with test_loss: 0.6271, epoch: 6
Train Epoch: [7/50] Loss: 1.4045 lco: 0.7894 lor: 0.0000 lsu: 0.6151 ldt: 0.0000 TrainAcc: 99.73 TestAcc: 98.59 TIME: 1.5300
Save model with test_loss: 0.6269, epoch: 7
Train Epoch: [8/50] Loss: 1.3650 lco: 0.7650 lor: 0.0000 lsu: 0.6000 ldt: 0.0000 TrainAcc: 99.92 TestAcc: 100.00 TIME: 1.5600
Save model with test_loss: 0.5729, epoch: 8
Train Epoch: [9/50] Loss: 1.3245 lco: 0.7190 lor: 0.0000 lsu: 0.6055 ldt: 0.0000 TrainAcc: 100.00 TestAcc: 96.82 TIME: 1.4700
Train Epoch: [10/50] Loss: 1.3165 lco: 0.7125 lor: 0.0000 lsu: 0.6039 ldt: 0.0000 TrainAcc: 100.00 TestAcc: 91.87 TIME: 1.4600
Train Epoch: [11/50] Loss: 1.2998 lco: 0.7145 lor: 0.0000 lsu: 0.5854

/home/icclab/miniconda3/envs/leo/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:198: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Train Epoch: [5/50] Loss: 1.4024 lco: 0.7892 lor: 0.0000 lsu: 0.6131 ldt: 0.0000 TrainAcc: 99.69 TestAcc: 92.93 TIME: 1.4600
Train Epoch: [6/50] Loss: 1.4211 lco: 0.8015 lor: 0.0000 lsu: 0.6196 ldt: 0.0000 TrainAcc: 99.33 TestAcc: 96.82 TIME: 1.4900
Save model with test_loss: 0.6064, epoch: 6
Train Epoch: [7/50] Loss: 1.3695 lco: 0.7758 lor: 0.0000 lsu: 0.5938 ldt: 0.0000 TrainAcc: 99.88 TestAcc: 97.17 TIME: 1.4600
Save model with test_loss: 0.6059, epoch: 7
Train Epoch: [8/50] Loss: 1.3974 lco: 0.7568 lor: 0.0000 lsu: 0.6407 ldt: 0.0000 TrainAcc: 99.80 TestAcc: 99.65 TIME: 1.4500
Save model with test_loss: 0.6032, epoch: 8
Train Epoch: [9/50] Loss: 1.3155 lco: 0.7104 lor: 0.0000 lsu: 0.6051 ldt: 0.0000 TrainAcc: 99.73 TestAcc: 98.94 TIME: 1.5200
Train Epoch: [10/50] Loss: 1.2942 lco: 0.6955 lor: 0.0000 lsu: 0.5987 ldt: 0.0000 TrainAcc: 100.00 TestAcc: 99.65 TIME: 1.5000
Save model with test_loss: 0.5718, epoch: 10
Train Epoch: [11/50] Loss: 1.3063 lco: 0.7014 lor: 0.0000 lsu: 0.6049 l

/home/icclab/miniconda3/envs/leo/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:198: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Train Epoch: [5/50] Loss: 1.5140 lco: 0.8633 lor: 0.0000 lsu: 0.6507 ldt: 0.0000 TrainAcc: 99.45 TestAcc: 94.35 TIME: 1.6600
Train Epoch: [6/50] Loss: 1.4201 lco: 0.8072 lor: 0.0000 lsu: 0.6129 ldt: 0.0000 TrainAcc: 99.76 TestAcc: 97.88 TIME: 1.6400
Save model with test_loss: 0.6110, epoch: 6
Train Epoch: [7/50] Loss: 1.4488 lco: 0.7988 lor: 0.0000 lsu: 0.6500 ldt: 0.0000 TrainAcc: 99.69 TestAcc: 100.00 TIME: 1.6900
Train Epoch: [8/50] Loss: 1.3621 lco: 0.7573 lor: 0.0000 lsu: 0.6048 ldt: 0.0000 TrainAcc: 99.88 TestAcc: 96.82 TIME: 1.8900
Train Epoch: [9/50] Loss: 1.3781 lco: 0.7800 lor: 0.0000 lsu: 0.5981 ldt: 0.0000 TrainAcc: 99.96 TestAcc: 96.47 TIME: 1.7100
Train Epoch: [10/50] Loss: 1.3375 lco: 0.7352 lor: 0.0000 lsu: 0.6023 ldt: 0.0000 TrainAcc: 99.92 TestAcc: 98.94 TIME: 1.6500
Save model with test_loss: 0.5867, epoch: 10
Train Epoch: [11/50] Loss: 1.3277 lco: 0.7069 lor: 0.0000 lsu: 0.6207 ldt: 0.0000 TrainAcc: 99.88 TestAcc: 100.00 TIME: 1.6600
Train Epoch: [12/50] Loss: 1.329